# Tutorial 01: Basic Compilation with qb-compiler

This tutorial introduces the core workflow of the `qb-compiler` package:

1. Building quantum circuits using the `QBCircuit` fluent API
2. Compiling circuits with `QBCompiler` targeting a specific backend
3. Understanding `CompileResult` metrics (depth, fidelity, pass log)
4. Estimating execution cost with `CostEstimator`

By the end you will be able to compile a circuit for any supported backend and interpret the compilation output.

## 1. Building a Circuit with QBCircuit

`QBCircuit` is the compiler's lightweight, vendor-neutral circuit representation. It stores an ordered list of `GateOp` instructions and provides fluent builder methods for common gates.

In [ ]:
from qb_compiler import QBCircuit, GateOp

In [ ]:
# Create a 2-qubit Bell state circuit using the fluent API.
# Each builder method (h, cx, measure_all) returns the circuit, so calls chain.
bell = QBCircuit(2).h(0).cx(0, 1).measure_all()

print(f"Circuit:      {bell}")
print(f"Num qubits:   {bell.n_qubits}")
print(f"Gate count:   {bell.gate_count}")
print(f"Depth:        {bell.depth}")
print(f"2Q gates:     {bell.two_qubit_count}")
print(f"Gate names:   {bell.gate_names}")

You can also add gates one at a time using the general `add()` method, which accepts a gate name, qubit tuple, and optional parameter tuple.

In [ ]:
import math

# Build a parameterised circuit manually
circ = QBCircuit(3)
circ.h(0)
circ.cx(0, 1)
circ.cx(1, 2)
circ.rz(2, math.pi / 4)   # RZ rotation by pi/4
circ.rx(1, math.pi / 2)   # RX rotation by pi/2
circ.x(0)                 # Pauli-X
circ.measure_all()

print(f"Circuit:    {circ}")
print(f"Depth:      {circ.depth}")
print(f"Gate count: {circ.gate_count}")
print()
print("Operations:")
for op in circ.ops:
    print(f"  {op.name:10s} qubits={op.qubits}  params={op.params}")

## 2. Supported Backends

Before compiling, let's see which backends are available. Each backend has a `BackendSpec` with hardware metadata: qubit count, basis gate set, median error rates, and cost per shot.

In [ ]:
from qb_compiler import BACKEND_CONFIGS

for name, spec in BACKEND_CONFIGS.items():
    print(
        f"{name:20s}  provider={spec.provider:12s}  "
        f"qubits={spec.n_qubits:4d}  "
        f"cx_err={spec.median_cx_error:.4f}  "
        f"$/shot={spec.cost_per_shot:.5f}"
    )

## 3. Compiling a Circuit

`QBCompiler` is the main entry point. The simplest way to create one is with `QBCompiler.from_backend()`, which configures the compiler for a specific device.

Available strategies:
- `"fidelity_optimal"` -- maximise estimated output fidelity (default, optimization level 3)
- `"depth_optimal"` -- minimise compiled circuit depth (optimization level 2)
- `"budget_optimal"` -- minimise cost while maintaining acceptable fidelity (optimization level 1)

In [ ]:
from qb_compiler import QBCompiler

# Create a compiler targeting IBM Fez with the default fidelity_optimal strategy
compiler = QBCompiler.from_backend("ibm_fez")

print(f"Strategy:           {compiler.strategy}")
print(f"Backend:            {compiler.config.backend}")
print(f"Optimization level: {compiler.config.optimization_level}")
print(f"Basis gates:        {compiler.config.effective_basis_gates}")

In [ ]:
# Compile the Bell state circuit
circuit = QBCircuit(2).h(0).cx(0, 1).measure_all()

result = compiler.compile(circuit)

print(f"Original depth:     {result.original_depth}")
print(f"Compiled depth:     {result.compiled_depth}")
print(f"Depth reduction:    {result.depth_reduction_pct:.1f}%")
print(f"Estimated fidelity: {result.estimated_fidelity:.4f}")
print(f"Compile time:       {result.compilation_time_ms:.2f} ms")

## 4. Inspecting the CompileResult

The `CompileResult` object contains everything about the compilation run: the compiled circuit, before/after metrics, estimated fidelity, and a detailed pass log.

In [ ]:
# The compiled circuit is also a QBCircuit
compiled = result.compiled_circuit

print(f"Compiled circuit:   {compiled}")
print(f"Gate count:         {compiled.gate_count}")
print(f"Gate names:         {compiled.gate_names}")
print()
print("Compiled operations:")
for op in compiled.ops:
    print(f"  {op.name:10s} qubits={op.qubits}  params={op.params}")

### The Pass Log

Each `PassResult` in the pass log records what a compiler pass did: gate count and depth before/after, and how long it took.

In [ ]:
print(f"{'Pass Name':25s}  {'Depth':>12s}  {'Gates':>12s}  {'Time (ms)':>10s}")
print("-" * 65)
for p in result.pass_log:
    print(
        f"{p.pass_name:25s}  "
        f"{p.depth_before:4d} -> {p.depth_after:<4d}  "
        f"{p.gate_count_before:4d} -> {p.gate_count_after:<4d}  "
        f"{p.elapsed_ms:8.3f}"
    )

## 5. Compiling a Larger Circuit

Let's try a more realistic circuit with multiple gate types and see the optimizer at work.

In [ ]:
# Build a 4-qubit circuit with redundant gates that the optimizer can cancel
circ = QBCircuit(4)

# Layer 1: Hadamards
for q in range(4):
    circ.h(q)

# Layer 2: entangling CX gates
circ.cx(0, 1)
circ.cx(2, 3)

# Layer 3: rotations
circ.rz(0, math.pi / 4)
circ.rz(1, math.pi / 8)
circ.rz(2, math.pi / 4)
circ.rz(3, math.pi / 8)

# Layer 4: more entangling
circ.cx(1, 2)

# Layer 5: redundant H-H pairs that should cancel
circ.h(0)
circ.h(0)  # cancels with previous H on qubit 0

# Layer 6: redundant CX-CX pair
circ.cx(0, 1)
circ.cx(0, 1)  # cancels with previous CX on same qubits

circ.measure_all()

print(f"Before: {circ}")
print(f"  Depth:      {circ.depth}")
print(f"  Gate count: {circ.gate_count}")
print(f"  2Q gates:   {circ.two_qubit_count}")

In [ ]:
result = compiler.compile(circ)

print(f"After compilation:")
print(f"  Depth:      {result.compiled_depth}  (was {result.original_depth})")
print(f"  Fidelity:   {result.estimated_fidelity:.4f}")
print(f"  Reduction:  {result.depth_reduction_pct:.1f}%")
print(f"  Time:       {result.compilation_time_ms:.2f} ms")
print()
print("Pass log:")
for p in result.pass_log:
    print(f"  {p.pass_name:25s}  gates {p.gate_count_before} -> {p.gate_count_after}")

## 6. Cost Estimation

The compiler can estimate execution cost based on the backend's per-shot pricing, circuit depth, and number of qubits.

In [ ]:
# Estimate cost for running the compiled circuit at various shot counts
compiled = result.compiled_circuit

for shots in [100, 1024, 4096, 10_000]:
    cost = compiler.estimate_cost(compiled, shots=shots)
    print(
        f"  {shots:>6,} shots  ->  "
        f"${cost.total_usd:.4f}  "
        f"(${cost.cost_per_shot_usd:.6f}/shot)  "
        f"backend={cost.backend}"
    )

### Budget Guard

You can pass a `budget_usd` parameter to `compile()`. If the estimated cost at 1024 shots exceeds the budget, a `BudgetExceededError` is raised.

In [ ]:
from qb_compiler import BudgetExceededError

circuit = QBCircuit(2).h(0).cx(0, 1).measure_all()

# This should succeed -- Bell state is very cheap
result = compiler.compile(circuit, budget_usd=1.00)
print(f"Compiled within budget: fidelity={result.estimated_fidelity:.4f}")

# This will raise -- the budget is impossibly small
try:
    compiler.compile(circuit, budget_usd=0.000001)
except BudgetExceededError as e:
    print(f"Budget exceeded: {e}")

## 7. Comparing Strategies

You can override the strategy per compilation call to compare results.

In [ ]:
# Build a circuit to compare strategies on
circ = QBCircuit(4)
for q in range(4):
    circ.h(q)
circ.cx(0, 1).cx(1, 2).cx(2, 3)
circ.rz(0, math.pi / 4).rz(1, math.pi / 4).rz(2, math.pi / 4).rz(3, math.pi / 4)
circ.cx(0, 1).cx(1, 2).cx(2, 3)
circ.measure_all()

print(f"Input: depth={circ.depth}, gates={circ.gate_count}")
print()

for strategy in ["fidelity_optimal", "depth_optimal", "budget_optimal"]:
    r = compiler.compile(circ, strategy=strategy)
    print(
        f"  {strategy:20s}  depth={r.compiled_depth:3d}  "
        f"fidelity={r.estimated_fidelity:.4f}  "
        f"time={r.compilation_time_ms:.2f}ms"
    )

## 8. Targeting Different Backends

Switching backends is as simple as changing the backend name. The compiler automatically selects the correct basis gates and error rates.

In [ ]:
circ = QBCircuit(3).h(0).cx(0, 1).cx(1, 2).measure_all()

for backend in ["ibm_fez", "ibm_torino", "rigetti_ankaa", "ionq_aria", "iqm_garnet"]:
    comp = QBCompiler.from_backend(backend)
    r = comp.compile(circ)
    cost = comp.estimate_cost(r.compiled_circuit, shots=1024)
    print(
        f"  {backend:18s}  depth={r.compiled_depth:3d}  "
        f"fidelity={r.estimated_fidelity:.4f}  "
        f"cost=${cost.total_usd:.4f}"
    )

## Summary

In this tutorial you learned:

- **QBCircuit** -- building circuits with the fluent API (`h()`, `cx()`, `rz()`, `measure_all()`, or the general `add()` method)
- **QBCompiler.from_backend()** -- creating a compiler for any supported backend
- **CompileResult** -- inspecting compiled depth, estimated fidelity, and the pass log
- **Cost estimation** -- using `estimate_cost()` and the `budget_usd` guard
- **Strategies** -- comparing `fidelity_optimal`, `depth_optimal`, and `budget_optimal`
- **Multi-backend** -- compiling the same circuit for different devices

Next up: [Tutorial 02 -- Calibration-Aware Compilation](02-calibration-aware.ipynb) shows how to use real calibration data for smarter qubit mapping.